# Collect multimer data into 1 MuData object and perform multimer annotation

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
import tempfile
from muon import prot as pt
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

#For the model:
from sklearn.mixture import GaussianMixture

import warnings
from pandas.errors import SettingWithCopyWarning

warnings.filterwarnings("ignore", category=SettingWithCopyWarning)

# Functions

In [ ]:
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRAC', 'TRB', 'TRDV', 'TRDC', 'TRG']

def filter_tcr_genes(adata):
    non_tcr_genes = adata.var_names
    for prefix in tcr_gene_prefixs:
        non_tcr_genes = [el for el in non_tcr_genes if not el.startswith(prefix)]
    adata = adata[:, non_tcr_genes]
    return adata

def filter_high_var(adata, n):
    #sc.pp.highly_variable_genes(
    #adata,
    #flavor="seurat_v3",
    #n_top_genes=n,
    #layer="counts",
    #batch_key="pool",
    #subset=True,
    #span = 0.75) #default = 0.3. It's the fraction of cells used for estimation of hvg - if I have 150 cells in one sample but 150.000 cells in total then this sample becomes an issue. 
    
    sc.pp.highly_variable_genes(adata, n_top_genes=n)
    adata = adata[:, adata.var['highly_variable']].copy()
    return adata


def calculate_umap(adata, n_high_var=5000, remove_tcr_genes=True):
    """
    Calculates UMAP coordinates
    :param adata: AnnData object
    :param n_high_var: how many variable genes to use, None uses raw data
    :param remove_tcr_genes: whether to remove TCR genes
    :return: results stored in adata object
    """
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)            
    sc.tl.umap(adata_tmp)
    adata.obsm['X_umap'] = adata_tmp.obsm['X_umap']


def calculate_leiden(adata, resolution, n_high_var=5000, remove_tcr_genes=True):
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)
    sc.tl.leiden(adata_tmp, resolution=resolution)

    leiden_column_name = f"leiden_{resolution}"
    adata.obs[leiden_column_name] = adata_tmp.obs['leiden']

## Load data

In [ ]:
mdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_multimer_dataset_raw.h5mu")


In [ ]:
mdata

### Clean up the 3 modalities

In [ ]:
sc.pp.calculate_qc_metrics(mdata["gex"], percent_top=None, log1p=False, inplace=True)

In [ ]:
# #All genes must be present with at least 2 cells before normalization
mu.pp.filter_var(mdata["gex"], 'n_cells_by_counts', lambda x: x > 1)

In [ ]:
###Clean-up.
#Make sure only the indexes recorded in GEX are used in the other modalities too
index_list = mdata["gex"].obs.index.tolist()

## define a cell index
mdata["airr"].obs["cell_index"] = mdata["airr"].obs.index
mdata["prot"].obs["cell_index"] = mdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
mdata["airr"].obs["filter_index"] = pd.Categorical(np.where(mdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
mdata["prot"].obs["filter_index"] = pd.Categorical(np.where(mdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that mdata["airr"].obs and mdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(mdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(mdata["prot"], 'filter_index', lambda x: x == "GEX_present")

In [ ]:
mdata.update()

In [ ]:
mdata

# Cell counts

In [ ]:
# Look at occurence of samples
df = pd.DataFrame(mdata["gex"].obs[["donor"]].value_counts()).reset_index()

plt.figure(figsize=(5, 4))

# Overlay dots (jittered)
sb.stripplot(
    data=df,
    x="donor",
    y="count",
    jitter=0.15,
    size=6,
    alpha=0.8,
    color="black"
)

plt.ylabel("Cell count")
plt.xlabel("Phenotype")
plt.xticks(rotation=45, ha="right")
plt.title("Cell counts by phenotype")
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5)) #moves legend

sb.despine()
plt.tight_layout()
plt.show()


### Define clone_handle

In [ ]:
# Define clone_handle - a combination of the donor information and the clone_id information to make each shared clone unique in each donor. 
mdata["gex"].obs["clone_handle"] = mdata["gex"].obs["donor"].astype(str) + "-" + mdata["gex"].obs["clone_id"].astype(str)

print(mdata["gex"].obs["clone_handle"].value_counts().head(5))

# If clone_ind is "NaN" then clone_handle should also be NaN
mdata["gex"].obs["clone_handle"] = np.where(mdata["gex"].obs["clone_id"].isna(), pd.NA, mdata["gex"].obs["clone_handle"])

# #Remove ".0" from the clone handle
# mdata["gex"].obs["clone_handle"] = (
#     mdata["gex"].obs["clone_handle"]
#     .astype(str)
#     .str.replace(r"\.0$", "", regex=True)  #\ escape     #.0 string to match    $ match end of string
# )

# If the edits worked, then na should not be the most counted (otherwise it will be)
print(mdata["gex"].obs["clone_handle"].value_counts().head(5))

mdata["airr"].obs["clone_handle"] = mdata["gex"].obs["clone_handle"].copy()


# Manual clone count

In [ ]:
## Calculate clone_size = number of rows per clone_id
mdata["gex"].obs["clone_size"] = mdata["gex"].obs.groupby("clone_handle", observed=True)["clone_handle"].transform("count")

# Define NA clones as 0 clone count
mdata["gex"].obs["clone_size"] = np.where(mdata["gex"].obs["clone_handle"].isna(), 0, mdata["gex"].obs["clone_size"])

## Share labels with airr
mdata["airr"].obs["clone_size"] = mdata["gex"].obs["clone_size"].copy()

mdata["gex"].obs[["clone_handle","clone_size"]].sort_values("clone_size", ascending=False).drop_duplicates()

## Multimer annotation

In [ ]:
mult_ann = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/FeatureBarcodCorrected.xlsx")
ListOfEpitopes = mult_ann["id"][18:].tolist()

#Place the list of epitopes heere
mdata["gex"].uns["epitopes"] = ListOfEpitopes

#Mimi also places the "epitope columns in .obsm"
mdata["gex"].obsm["epitopes"] = mdata["gex"].obs[ListOfEpitopes]

In [ ]:
#Normalize each epitope in the .obs object. 
for ep in mdata["gex"].uns['epitopes']:
    mdata["gex"].obs[f'log_{ep}'] = np.log(mdata["gex"].obs[ep]+1)

In [ ]:
mdata["gex"].obs['n_count_dextramer'] = np.sum(mdata["gex"].obsm['epitopes'], axis=1)
mdata["gex"].obs['n_max_dextramer'] = np.max(mdata["gex"].obsm['epitopes'], axis=1)
mdata["gex"].obs['max_dextramer'] = np.nanargmax(mdata["gex"].obsm['epitopes'], axis=1)
mdata["gex"].obs['max_dextramer'] = mdata["gex"].obs['max_dextramer'].apply(lambda x: mdata["gex"].uns['epitopes'][x])
mdata["gex"].obs['max_dextramer'][mdata["gex"].obs['n_max_dextramer']==0] = 'NONE'
mdata["gex"].obs['%_max_dextramer'] = mdata["gex"].obs['n_max_dextramer'] / mdata["gex"].obs['n_count_dextramer']

# mdata["gex"].obs['clone_id'] = mdata["airr"].obs['clone_id']

In [ ]:
mdata['gex'].obs['dataset'] = mdata['gex'].obs['batch']

## Specificity annotation

### Clone Purity over all cells before any cutoff

In [ ]:
#plot clone purity (before UMI filtering)
df = pd.concat([mdata["gex"].obs['n_count_dextramer'], mdata["gex"].obs['n_max_dextramer'], 
                mdata["gex"].obs['%_max_dextramer'], mdata["gex"].obs['max_dextramer'],
                mdata["gex"].obs['clone_handle']], axis=1)

for i in mdata["gex"].uns['epitopes']:
    df[i] = 0
    df[i][df['max_dextramer']==i] = 1

df = df.drop(['max_dextramer'], axis=1)
dff = df.groupby('clone_handle').mean().reset_index()

#plot clone id versus epitope purity
for i in mdata["gex"].uns['epitopes']:
    fig, ax = plt.subplots(figsize=(5, 2))
    y=dff[i].sort_values().to_list()
    plt.plot(y)
    plt.title(i)
    plt.show()
    
#due to "mean" the number for the epitopes now equals the mean amount of positive cells per clone

## Continue with a subset of data - or all data - here with all data

#### Predictions with variable clone purity

In [ ]:
mult_ann = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/FeatureBarcodCorrected.xlsx")
ListOfEpitopes = mult_ann["id"][18:].tolist()

In [ ]:
#Specificity prediction with variable clonepurity
ep = ListOfEpitopes

clonepurity = np.arange(0.01, 1.01, 0.025)

new_list = ListOfEpitopes + ["clone_purity"]
x= pd.DataFrame(columns=new_list, index=range(len(clonepurity)))

#choose only data from first experiment
data = mdata

#generate the table for all cells and epitopes
z = pd.concat([data["gex"].obs[ep],
                   data["gex"].obs['n_count_dextramer'],
                   data["gex"].obs['n_max_dextramer'],
                   data["gex"].obs['max_dextramer'],
                   data["gex"].obs['%_max_dextramer'],
                   data["gex"].obs['clone_handle'],
                   data["gex"].obs['clone_id'],
                   data["gex"].obs['clone_size']], axis=1)

clonepurity #Increments of 0.025 bebtween 0.01 and 1.01

In [ ]:
#remove cells that do not belong to any clone (clone_id = Nan)
z = z[~z['clone_id'].isna()]

#start the loop for going through the epitopes
for i in range(len(ep)):
    z['max'] = 0
    z['max_dextramer'] = data["gex"].obs['max_dextramer'] 
    z[ep[i]+'_pur'] = z[ep[i]]/z['n_count_dextramer'] 
    z[ep[i]+'_keep'] = 'no' 
    z[ep[i]+'_prediction'] = 'no'

    #train model on clones with more than 5 cells
    X = np.array((z[z['clone_size']>=5][[ep[i],ep[i]+'_pur']]).dropna())
    gm = GaussianMixture(n_components=2, random_state=0).fit(X)
    
    #to make sure that 1 always corresponds to the higher values
    if (gm.means_[0][0] < gm.means_[1][0]):
        print('ok')
    else:
        print('ohoh')
        
  
    #use model to filter (predict by cell purity and UMI counts, exclude cells with less than 3 counts)
    for c in range(len(z)):
        if z[ep[i]][c] is not None and np.any(z[ep[i]][c]):
            if (gm.means_[0][0] < gm.means_[1][0]):
                if ((z[ep[i]][c] >= 0) & (z[ep[i]+'_pur'][c] >= 0) & (z['n_count_dextramer'][c] > 3)):
                    z[ep[i]+'_prediction'][c] = gm.predict([[z[ep[i]][c], z[ep[i]+'_pur'][c]]])
                else:
                    z[ep[i]+'_prediction'][c] = 2 
            else:
                if ((z[ep[i]][c] >= 0) & (z[ep[i]+'_pur'][c] >= 0) & (z['n_count_dextramer'][c] > 3)):
                    if gm.predict([[z[ep[i]][c], z[ep[i]+'_pur'][c]]]) == 0:
                        z[ep[i]+'_prediction'][c] = 1
                    else:
                        z[ep[i]+'_prediction'][c] = 0
                else:
                    z[ep[i]+'_prediction'][c] = 2 #cells with no prediction
        
    #filter by variable clone purities and note how many clones would still be left
    z['max'][z['max_dextramer'] != ep[i]] = 0 
    z['max'][z[ep[i]+'_prediction'] == 0] = 0 
    z['max'][(z[ep[i]].isna() == True) & (z[ep[i]+'_prediction'] == 2)] = np.nan 
    z['max'][(z[ep[i]].isna() == False) & (z[ep[i]+'_prediction'] == 2)] = 0
    z['max'][(z['max_dextramer'] == ep[i]) & (z[ep[i]+'_prediction'] == 1)] = 1 

    for p in range(len(clonepurity)):
        y = pd.DataFrame(z.groupby('clone_size')['max'].mean()) 
        goodclones = (y[y['max']>clonepurity[p]]).index
        x['clone_purity'][p] = clonepurity[p]
        x[ep[i]][p]= len(goodclones)

In [ ]:
for i in range(len(ep)):
    plt.figure(figsize=(3, 2))  # width=4, height=3 inches
    sb.lineplot(data=x, x='clone_purity', y=ep[i])
    plt.show()

In [ ]:
# fig, ax = plt.subplots()
# sb.set_palette('husl',13)
# for i in range(len(ep)):
#     sb.lineplot(data=x, x='clone_purity', y=ep[i], ax=ax, label=ep[i])
# ax.set_xlim(0, 1)
# ax.legend(loc='upper left', bbox_to_anchor=(1, 1), markerscale=0.5, handlelength=1.5)
# ax.set_xlabel('Clone Purity')
# ax.set_ylabel('Number of Clones')
# ax.axvline(x=0.6, color='black', linestyle='--')
# plt.show()

#### Prediction with Clone Purity set to 0.6

In [ ]:
#Specificity prediction with set clonepurity
ep = ListOfEpitopes

clonepurity = 0.6

#choose only data from first experiment
data = mdata

#generate the table for all cells and epitopes
z = pd.concat([data["gex"].obs[ep],
                   data["gex"].obs['n_count_dextramer'],
                   data["gex"].obs['n_max_dextramer'],
                   data["gex"].obs['max_dextramer'],
                   data["gex"].obs['%_max_dextramer'],
                   data["gex"].obs['clone_id'],
                   data["gex"].obs['clone_handle'],
                   data["gex"].obs['clone_size']], axis=1)

#remove cells that do not belong to any clone (clone_id = Nan) --> account for this case later to save also nan cells
#z = z[z['clone_id']>=0]

#start the loop for going through the epitopes
for i in range(len(ep)):
    z['max'] = 0
    z['max_dextramer'] = data["gex"].obs['max_dextramer'] #need to add this column every round as it gets changed
    
    z[ep[i]+'_pur'] = z[ep[i]]/z['n_count_dextramer'] #cell purity for every epitope
    z[ep[i]+'_keep'] = 'no' 
    z[ep[i]+'_prediction'] = 'no'

    #train model on clones with more than 5 cells
    X = np.array((z[z['clone_size']>=5][[ep[i],ep[i]+'_pur']]).dropna())
    gm = GaussianMixture(n_components=2, random_state=0).fit(X)
    
    #to make sure that 1 always corresponds to the higher values
    if (gm.means_[0][0] < gm.means_[1][0]):
        print('ok')
    else:
        print('ohoh')
    
    #use model to filter (predict by cell purity and UMI counts, exclude cells with less than 3 counts)
    for c in range(len(z)):
        if z[ep[i]][c] is not None and np.any(z[ep[i]][c]):
            if (gm.means_[0][0] < gm.means_[1][0]):
                if ((z[ep[i]][c] >= 0) & (z[ep[i]+'_pur'][c] >= 0) & (z['n_count_dextramer'][c] > 3)):
                    z[ep[i]+'_prediction'][c] = gm.predict([[z[ep[i]][c], z[ep[i]+'_pur'][c]]])
                else:
                    z[ep[i]+'_prediction'][c] = 2 
            else:
                if ((z[ep[i]][c] >= 0) & (z[ep[i]+'_pur'][c] >= 0) & (z['n_count_dextramer'][c] > 3)):
                    if gm.predict([[z[ep[i]][c], z[ep[i]+'_pur'][c]]]) == 0:
                        z[ep[i]+'_prediction'][c] = 1
                    else:
                        z[ep[i]+'_prediction'][c] = 0
                else:
                    z[ep[i]+'_prediction'][c] = 2 #cells with no prediction
        
    #filter by clone purity - to calculate the clone purity set cells to be counted specific (1) or not specific (0)
    #or not counted at all (nan)
    
    z['max'][z['max_dextramer'] != ep[i]] = 0 #cells with another maximum dextramer don't contribute to clone purity 
    z['max'][z[ep[i]+'_prediction'] == 0] = 0 #cells that are predicted not to be specific don't contribute to clone purity
    z['max'][(z[ep[i]].isna() == True) & (z[ep[i]+'_prediction'] == 2)] = np.nan #cells with no dextramer count and no prediction are not counted
    z['max'][(z[ep[i]].isna() == False) & (z[ep[i]+'_prediction'] == 2)] = 0 #cells with dextramer counts but no prediction do not contribute to clone purity
    z['max'][(z['max_dextramer'] == ep[i]) & (z[ep[i]+'_prediction'] == 1)] = 1 #cells with positive prediction do contribute to clone purity
  
    y = pd.DataFrame(z[~z['clone_handle'].isna()].groupby('clone_handle')['max'].mean()) #the mean values equal clone purity
    goodclones = (y[y['max']>=clonepurity]).index #list of clones to include
    badclones = (y[y['max']<clonepurity]).index #list of clones to exclude

    #generate input for "keep" column
    z[ep[i]+'_keep'][z[ep[i]+'_prediction']==1] = 'yes_prediction' #cells to count as specific due to their prediction
    z[ep[i]+'_keep'][z[ep[i]+'_prediction']==0] = 'no_prediction' #cells that are unspecific according to the prediction
    z[ep[i]+'_keep'][z[ep[i]+'_prediction']==2] = 'not_predicted' #cells without prediction
    #take clone purity into account
    z[ep[i]+'_keep'][(z['clone_handle'].isin(goodclones)) & (z[ep[i]+'_keep'] != 'yes_prediction')] = 'yes_clone' #include cells if the rest of their clone is ok
    z[ep[i]+'_keep'][(z['clone_handle'].isin(badclones)) & (z[ep[i]+'_keep'] == 'yes_prediction')] = 'no_clone' 

#### Stripplots

In [ ]:
pal = {'yes_prediction' : 'green', 
       'no_prediction' : 'black',
       'not_predicted' : 'darkgrey',
       'yes_clone': 'lightgreen',
       'no_clone' : 'grey',
       'no': 'yellow'}

In [ ]:
#which clones are in exp1?
clones = (mdata['gex'].obs['clone_id'][mdata['gex'].obs['batch']=='NK13_1']).unique()
len(clones)

In [ ]:
z.columns

In [ ]:
# #plot the clones
# sb.set(rc={'figure.figsize':(27,5)})
# sb.set_style("whitegrid")

# lower = range(0, 3218, 30)

# upper = range(30, 3218, 30)

# for i,j in zip(lower,upper):
#     sb.stripplot(data=z[(z['clone_id']>=clones[i])&(z['clone_id']<clones[j])], 
#              x=z['clone_id'][(z['clone_id']>=clones[i])&(z['clone_id']<clones[j])], 
#              y=z['sc-1OS_19'][(z['clone_id']>=clones[i])&(z['clone_id']<clones[j])], 
#              hue=z['sc-1OS_19_keep'][(z['clone_id']>=clones[i])&(z['clone_id']<clones[j])], palette=pal, legend=False, jitter=True)
#     plt.figsize=(150,10)
#     # plt.savefig(f'/media/agschober/HDD12/3_scRNA-Seq_Sina/3_Specificity/Stripplots/Exp1_NS4B214_stripplots_clone_{i}-{j}.png', dpi=600)
#     plt.show()

### Annotate the specific cells in mdata

In [ ]:
#select cells to annotate in data - rather cells than clones, otherwise the NAN clone cells are lost again
#mark whether NS4B214 was annotated due to prediction or clone

mdata["gex"].obs['specific'] = 'no_binding'

# mdata["gex"].obs['specific'][(mdata["gex"].obs.index.isin(z.index[z['NS4B214_keep'].isin(['yes_prediction'])]))&
#                              (mdata["gex"].obs['dataset']=='exp1')] = 'NS4B214'

# mdata["gex"].obs['specific'][(mdata["gex"].obs.index.isin(z.index[z['NS4B214_keep'].isin(['yes_clone'])]))&
#                              (mdata["gex"].obs['dataset']=='exp1')] = 'NS4B214_c'

#loop over other specificities
epitopes = ListOfEpitopes

for e in epitopes:
    mdata["gex"].obs['specific'][(mdata["gex"].obs.index.isin(z.index[z[e+'_keep'].isin(['yes_prediction', 'yes_clone'])]))] = e

In [ ]:
mdata["gex"].obs

## Save the data

In [ ]:
z.to_csv("../../04_data_objects/06_preprocessed_v2/multimer_pred_z_df.csv")

In [ ]:
mdata.write(filename="../../04_data_objects/06_preprocessed_v2/2026_6_multimer_dataset_preprocessed.h5mu")